# Exploratory Analysis — Supply Chain Shortage Evaluation Framework

This notebook documents the first stage of the project: understanding the M5 Forecasting dataset and deciding how to represent the data for a shortage-risk evaluation framework.

The key methodological point is that the dataset does not contain direct inventory or stockout labels. Because of that, we begin by studying sales behavior and defining a proxy target for **inventory stress** based on unusually high demand.


## 1. Environment and package check

Before touching the data, we verify that the core Python packages load correctly. This confirms that the local VS Code/Jupyter environment is ready for analysis.

At this stage, we are not modeling yet. We are only checking that the notebook runtime can import the libraries needed for data analysis and later baseline modeling.


In [1]:
import pandas as pd
import numpy as np
import sklearn
import xgboost

print("All packages loaded")

All packages loaded


## 2. Load the raw M5 dataset files

The dataset is stored in `data/raw/`. We load three core files:

- `calendar.csv`: maps day numbers like `d_1`, `d_2`, etc. to actual dates, events, and SNAP indicators.
- `sell_prices.csv`: contains item prices by store and week.
- `sales_train_validation.csv`: contains daily unit sales for each item-store combination.

The `.shape` output gives a first sense of scale: number of rows and columns in each file.


In [2]:
from pathlib import Path

RAW_DIR = Path("../data/raw")

calendar = pd.read_csv(RAW_DIR / "calendar.csv")
prices = pd.read_csv(RAW_DIR / "sell_prices.csv")
sales = pd.read_csv(RAW_DIR / "sales_train_validation.csv")

print("calendar:", calendar.shape)
print("prices:", prices.shape)
print("sales:", sales.shape)

calendar: (1969, 14)
prices: (6841121, 4)
sales: (30490, 1919)


## 3. Inspect column structure

We inspect column names to understand how the dataset is organized.

The sales file is in **wide format**: one row per item-store combination, with each day stored as a separate column (`d_1` through `d_1913`). This is convenient for compact storage, but less convenient for feature engineering and machine learning pipelines.


In [3]:
print("Calendar columns:")
print(calendar.columns.tolist())

print("\nPrices columns:")
print(prices.columns.tolist())

print("\nSales columns:")
print(sales.columns.tolist()[:20])
print("...")
print(sales.columns.tolist()[-10:])

Calendar columns:
['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']

Prices columns:
['store_id', 'item_id', 'wm_yr_wk', 'sell_price']

Sales columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1', 'd_2', 'd_3', 'd_4', 'd_5', 'd_6', 'd_7', 'd_8', 'd_9', 'd_10', 'd_11', 'd_12', 'd_13', 'd_14']
...
['d_1904', 'd_1905', 'd_1906', 'd_1907', 'd_1908', 'd_1909', 'd_1910', 'd_1911', 'd_1912', 'd_1913']


## 4. Understand dataset hierarchy

We count the number of unique items, stores, categories, and departments.

This tells us the product hierarchy and retail coverage of the dataset. It also helps us decide whether to model at the item level, store level, category level, or some sampled subset during early experimentation.


In [4]:
print("Unique items:", sales["item_id"].nunique())
print("Unique stores:", sales["store_id"].nunique())
print("Unique categories:", sales["cat_id"].nunique())
print("Unique departments:", sales["dept_id"].nunique())

Unique items: 3049
Unique stores: 10
Unique categories: 3
Unique departments: 7


## 5. Preview the sales table

We preview the first few rows to see the actual layout of the sales data.

Each row represents a single item-store series. The daily sales values are spread across many columns, which is why the next step reshapes the table.


In [5]:
sales.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


## 6. Reshape sales from wide format to long format

We use `pandas.melt()` to convert the table from wide format to long format.

Before melting:

- One row = one item-store combination
- Each day is a separate column

After melting:

- One row = one item-store-day observation
- The day is stored in a `day` column
- The daily unit sales value is stored in a `sales` column

This long format is much easier for building features such as rolling averages, lagged sales, demand spikes, and future stress-event labels.


In [6]:
sales_long = sales.melt(
    id_vars=[
        "id",
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id"
    ],
    var_name="day",
    value_name="sales"
)

sales_long.head()

,id,item_id,dept_id,cat_id,store_id,state_id,day,sales
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


## 7. Measure full long-format size and memory cost

After melting, we check the resulting shape and memory usage.

This is an important data engineering step. The full long-format dataset becomes very large, so we need to decide whether to continue with all rows or create a smaller development subset.


In [7]:
sales_long.shape
sales_long.memory_usage(deep=True).sum() / 1024**3

np.float64(22.782371260225773)

## 8. Create a manageable development subset

The full melted dataset used about 22.8 GB of memory, which is too large for comfortable experimentation.

To develop the evaluation methodology efficiently, we select the first 100 unique items and keep their store-level sales rows. This gives us a representative but much smaller working dataset.

This is not the final research claim. It is a practical development subset used to design and test the pipeline before scaling.


In [8]:
sample_items = sales["item_id"].unique()[:100]

sales_small = sales[
    sales["item_id"].isin(sample_items)
]

## 9. Check the sampled wide-format dataset size

We verify the size of the sampled dataset before reshaping it.

The expected structure is roughly 100 items across 10 stores, with the same daily sales columns as the full dataset.


In [9]:
sales_small.shape

(1000, 1919)

## 10. Melt the sampled dataset

We reshape only the sampled dataset into long format.

This gives us a much smaller item-store-day table that is practical for exploration, target definition, and early baseline experiments.


In [10]:
sales_small_long = sales_small.melt(
    id_vars=[
        "id",
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id"
    ],
    var_name="day",
    value_name="sales"
)

## 11. Check sampled long-format row count

We confirm the number of rows in the sampled long-format dataset.

This tells us how many item-store-day observations we can use for early feature engineering and target creation.


In [11]:
sales_small_long.shape

(1913000, 8)

## 12. Check sampled memory usage

We measure the memory footprint of the sampled long-format dataset.

This helps confirm that the subset is suitable for interactive notebook work in VS Code.


In [12]:
sales_small_long.memory_usage(deep=True).sum() / 1024**3

np.float64(0.7490312047302723)

## 13. Delete the full melted dataframe

Since the full `sales_long` dataframe consumed too much memory, we delete it after measuring its size.

This keeps the notebook responsive and prevents later cells from running out of memory.


In [13]:
del sales_long

## 14. Trigger garbage collection

After deleting the large dataframe, we manually run Python garbage collection.

This asks Python to reclaim unused memory so the notebook remains stable during further analysis.


In [14]:
import gc
gc.collect()

0

## 15. Reconfirm sampled wide-format size

We re-check `sales_small.shape` after freeing memory to confirm that our smaller working dataset is still available.


In [15]:
sales_small.shape

(1000, 1919)

## 16. Reconfirm sampled long-format size

We re-check `sales_small_long.shape` to verify that the smaller long-format table is still available for analysis.


In [16]:
sales_small_long.shape


(1913000, 8)

## 17. Reconfirm sampled memory usage

We measure memory again to ensure the working dataset remains manageable.

This reinforces the key project decision: develop the methodology on a smaller subset first, then scale later if needed.


In [17]:
sales_small_long.memory_usage(deep=True).sum() / 1024**3

np.float64(0.7490312047302723)

## 18. Preview sampled long-format data

We inspect the first few rows of the sampled long-format dataset.

At this point, each row should represent one item-store-day observation with columns for product identifiers, store identifiers, day, and sales.


In [18]:
sales_small_long.head()

,id,item_id,dept_id,cat_id,store_id,state_id,day,sales
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


## 19. Explore sales distribution by item

We summarize daily sales distribution for each item.

This helps us understand what “normal” and “unusually high” sales look like for each product. This matters because different products have very different typical sales volumes.


In [19]:
sales_small_long.groupby("item_id")["sales"].describe()

,count,mean,std,min,25%,50%,75%,max
item_id,,,,,,,,
HOBBIES_1_001,19130.0,0.213957,0.576033,0.0,0.0,0.0,0.0,6.0
HOBBIES_1_002,19130.0,0.264454,0.593988,0.0,0.0,0.0,0.0,11.0
HOBBIES_1_003,19130.0,0.075013,0.323133,0.0,0.0,0.0,0.0,6.0
HOBBIES_1_004,19130.0,2.047831,2.668301,0.0,0.0,1.0,3.0,25.0
HOBBIES_1_005,19130.0,0.764297,1.225230,0.0,0.0,0.0,1.0,15.0
...,...,...,...,...,...,...,...,...
HOBBIES_1_099,19130.0,0.791375,1.207007,0.0,0.0,0.0,1.0,9.0
HOBBIES_1_100,19130.0,0.396864,0.694519,0.0,0.0,0.0,1.0,7.0
HOBBIES_1_102,19130.0,0.101045,0.360911,0.0,0.0,0.0,0.0,8.0


## 20. Define initial demand-stress thresholds

Because the M5 dataset does not include inventory levels or true stockout labels, we define a proxy target.

Initial proxy definition:

> A demand-stress event occurs when daily sales exceed the item's historical 90th percentile.

This threshold identifies unusually high demand days for each item. It is not a perfect shortage label, but it is a transparent and reproducible proxy for inventory pressure.

The next step will be to convert these thresholds into a binary `stress_event` label.


In [20]:
stress_thresholds = (
    sales_small_long
    .groupby("item_id")["sales"]
    .quantile(0.90)
)

In [21]:
sales_small_long["stress_threshold"] = (
    sales_small_long["item_id"]
    .map(stress_thresholds)
)

In [22]:
sales_small_long["stress_event"] = (
    sales_small_long["sales"]
    > sales_small_long["stress_threshold"]
).astype(int)

In [23]:
sales_small_long[
    ["item_id", "sales", "stress_threshold", "stress_event"]
].head(20)

,item_id,sales,stress_threshold,stress_event
0,HOBBIES_1_001,0,1.0,0
1,HOBBIES_1_002,0,1.0,0
2,HOBBIES_1_003,0,0.0,0
3,HOBBIES_1_004,0,6.0,0
4,HOBBIES_1_005,0,2.0,0
5,HOBBIES_1_006,0,2.0,0
6,HOBBIES_1_007,0,1.0,0
7,HOBBIES_1_008,12,12.0,0
8,HOBBIES_1_009,2,2.0,0
9,HOBBIES_1_010,0,2.0,0


In [24]:
sales_small_long["stress_event"].value_counts()

stress_event
0    1796728
1     116272
Name: count, dtype: int64

In [25]:
sales_small_long["stress_event"].value_counts(normalize=True)

stress_event
0    0.93922
1    0.06078
Name: proportion, dtype: float64

In [26]:
sales_small_long.groupby("stress_event")["sales"].describe()

,count,mean,std,min,25%,50%,75%,max
stress_event,,,,,,,,
0,1796728.0,0.450217,1.135007,0.0,0.0,0.0,0.0,12.0
1,116272.0,5.493902,6.843242,1.0,2.0,3.0,6.0,160.0


In [27]:
sales_small_long["day"].head()

0    d_1
1    d_1
2    d_1
3    d_1
4    d_1
Name: day, dtype: object

In [28]:
sales_small_long["day_num"] = (
    sales_small_long["day"]
    .str.replace("d_", "")
    .astype(int)
)

In [29]:
sales_small_long = sales_small_long.sort_values(
    ["item_id", "day_num"]
)

In [30]:
sales_small_long["sales_lag_1"] = (
    sales_small_long
    .groupby("item_id")["sales"]
    .shift(1)
)

In [31]:
sales_small_long[
    ["item_id", "day", "day_num", "sales", "sales_lag_1"]
].head(20)

,item_id,day,day_num,sales,sales_lag_1
0,HOBBIES_1_001,d_1,1,0,NaN
100,HOBBIES_1_001,d_1,1,0,0.0
200,HOBBIES_1_001,d_1,1,0,0.0
300,HOBBIES_1_001,d_1,1,0,0.0
400,HOBBIES_1_001,d_1,1,0,0.0
500,HOBBIES_1_001,d_1,1,0,0.0
600,HOBBIES_1_001,d_1,1,0,0.0
700,HOBBIES_1_001,d_1,1,0,0.0
800,HOBBIES_1_001,d_1,1,0,0.0
900,HOBBIES_1_001,d_1,1,0,0.0


In [32]:
sales_small_long[
    (sales_small_long["sales"] > 3) &
    (sales_small_long["sales_lag_1"] > 3)
][
    ["item_id", "day", "day_num", "sales", "sales_lag_1", "stress_event"]
].head(30)

,item_id,day,day_num,sales,sales_lag_1,stress_event
1785102,HOBBIES_1_003,d_1786,1786,4,5.0,1
29503,HOBBIES_1_004,d_30,30,5,4.0,0
35203,HOBBIES_1_004,d_36,36,4,7.0,0
53503,HOBBIES_1_004,d_54,54,4,4.0,0
71503,HOBBIES_1_004,d_72,72,7,5.0,1
86603,HOBBIES_1_004,d_87,87,4,4.0,0
92203,HOBBIES_1_004,d_93,93,5,11.0,0
105503,HOBBIES_1_004,d_106,106,7,6.0,1
105603,HOBBIES_1_004,d_106,106,4,7.0,0
106003,HOBBIES_1_004,d_107,107,5,7.0,0


In [33]:
sales_small_long.groupby("stress_event")["sales_lag_1"].describe()

,count,mean,std,min,25%,50%,75%,max
stress_event,,,,,,,,
0,1796635.0,0.723655,2.276998,0.0,0.0,0.0,1.0,160.0
1,116265.0,1.268748,3.199349,0.0,0.0,0.0,1.0,77.0


In [34]:
sales_small_long["sales_lag_7"] = (
    sales_small_long
    .groupby("item_id")["sales"]
    .shift(7)
)

In [35]:
sales_small_long["rolling_mean_7"] = (
    sales_small_long
    .groupby("item_id")["sales"]
    .transform(
        lambda x: x.shift(1).rolling(7).mean()
    )
)

In [36]:
sales_small_long[
    [
        "item_id",
        "day",
        "sales",
        "sales_lag_1",
        "sales_lag_7",
        "rolling_mean_7",
        "stress_event"
    ]
].head(20)

,item_id,day,sales,sales_lag_1,sales_lag_7,rolling_mean_7,stress_event
0,HOBBIES_1_001,d_1,0,NaN,NaN,NaN,0
100,HOBBIES_1_001,d_1,0,0.0,NaN,NaN,0
200,HOBBIES_1_001,d_1,0,0.0,NaN,NaN,0
300,HOBBIES_1_001,d_1,0,0.0,NaN,NaN,0
400,HOBBIES_1_001,d_1,0,0.0,NaN,NaN,0
500,HOBBIES_1_001,d_1,0,0.0,NaN,NaN,0
600,HOBBIES_1_001,d_1,0,0.0,NaN,NaN,0
700,HOBBIES_1_001,d_1,0,0.0,0.0,0.0,0
800,HOBBIES_1_001,d_1,0,0.0,0.0,0.0,0
900,HOBBIES_1_001,d_1,0,0.0,0.0,0.0,0


In [37]:
sales_small_long.groupby("stress_event")[
    ["sales_lag_1", "sales_lag_7", "rolling_mean_7"]
].mean()

,sales_lag_1,sales_lag_7,rolling_mean_7
stress_event,,,
0,0.723655,0.732060,0.731914
1,1.268748,1.137186,1.139580


## Stress Threshold Methodology

We define a stress threshold per item using the 90th percentile of historical sales. This gives each item its own definition of unusually high demand.

Why not average? Because averages can be distorted by spikes. Percentiles are better for detecting unusual behavior.


## Stress Event Label

A stress event is defined as:

`sales > stress_threshold`

This converts raw sales into a binary target:
- 0 = normal demand
- 1 = high-demand stress


## Temporal Cleanup (`day_num`)

The `day` column was stored as text (`d_1`, `d_2`, ...). We converted it into numeric form so sorting follows true chronological order.


## Lag Features

Lag features capture past demand:
- `sales_lag_1`: yesterday's sales
- `sales_lag_7`: one week ago

These allow forecasting without data leakage.


## Rolling Mean Feature

`rolling_mean_7` captures average demand over the previous 7 days. This smooths noise and detects broader trends.


## Feature Sanity Check

We compare average feature values between normal and stress days. This helps verify whether our features actually contain predictive signal before training models.


In [38]:
feature_cols = [
    "sales_lag_1",
    "sales_lag_7",
    "rolling_mean_7"
]

model_data = sales_small_long.dropna(
    subset=feature_cols
)

X = model_data[feature_cols]
y = model_data["stress_event"]

print(X.shape)
print(y.shape)
print(y.value_counts(normalize=True))

(1912300, 3)
(1912300,)
stress_event
0    0.939224
1    0.060776
Name: proportion, dtype: float64


In [39]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

print(X_train.shape)
print(X_test.shape)

(1529840, 3)
(382460, 3)


In [40]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000
)

model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [41]:
y_pred = model.predict(X_test)

In [42]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      1.00      0.97    358598
           1       0.14      0.00      0.00     23862

    accuracy                           0.94    382460
   macro avg       0.54      0.50      0.48    382460
weighted avg       0.89      0.94      0.91    382460



In [43]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.81      0.87    358598
           1       0.10      0.30      0.14     23862

    accuracy                           0.78    382460
   macro avg       0.52      0.56      0.51    382460
weighted avg       0.89      0.78      0.83    382460



## Baseline Logistic Regression Results

We trained a first baseline logistic regression model using temporal lag features and rolling averages.
This establishes a simple benchmark before trying more complex models like Random Forest.


## Why Accuracy Was Misleading

The unweighted model achieved ~94% accuracy but almost completely failed to detect stress events.
This happened because the dataset is imbalanced (about 94% normal days).
This demonstrates that accuracy can hide poor rare-event performance.


## Class Weighting (`class_weight='balanced'`)

Class weighting increases the penalty for mistakes on the minority class.
It does not create new rows or duplicate data.
It only changes how expensive errors are during training.

This improved recall for stress events significantly.


## Domain Tradeoff: Supply Chain Perspective

**False Positive:** Predict shortage → extra inventory ordered.
Costly but usually manageable.

**False Negative:** Miss real shortage → stockout occurs.
This can lead to lost sales and customer dissatisfaction.

Because of this, recall is often prioritized over precision in shortage prediction.


In [44]:
from sklearn.ensemble import RandomForestClassifier

In [45]:
from sklearn.metrics import classification_report

In [46]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print(classification_report(y_test, rf_pred))

              precision    recall  f1-score   support

           0       0.98      0.35      0.52    358598
           1       0.08      0.87      0.15     23862

    accuracy                           0.38    382460
   macro avg       0.53      0.61      0.33    382460
weighted avg       0.92      0.38      0.49    382460



In [47]:
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.feature_importances_
})

feature_importance.sort_values(
    by="importance",
    ascending=False
)

,feature,importance
2,rolling_mean_7,0.693977
0,sales_lag_1,0.188923
1,sales_lag_7,0.117100


In [48]:
sales_small_long["rolling_std_7"] = (
    sales_small_long
    .groupby("item_id")["sales"]
    .transform(
        lambda x: x.shift(1).rolling(7).std()
    )
)

In [50]:
# updating feature cols:
feature_cols = [
    "sales_lag_1",
    "sales_lag_7",
    "rolling_mean_7",
    "rolling_std_7"
]

model_data = sales_small_long.dropna(
    subset=feature_cols
)

In [51]:
model_data = sales_small_long.dropna(
    subset=feature_cols
)

X = model_data[feature_cols]
y = model_data["stress_event"]

print(X.shape)
print(y.shape)

(1912300, 4)
(1912300,)


In [52]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

In [53]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print(classification_report(y_test, rf_pred))

              precision    recall  f1-score   support

           0       0.97      0.36      0.52    358598
           1       0.08      0.86      0.15     23862

    accuracy                           0.39    382460
   macro avg       0.53      0.61      0.34    382460
weighted avg       0.92      0.39      0.50    382460



In [54]:
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.feature_importances_
})

feature_importance.sort_values(
    by="importance",
    ascending=False
)

,feature,importance
2,rolling_mean_7,0.479916
3,rolling_std_7,0.408012
0,sales_lag_1,0.093113
1,sales_lag_7,0.018959
